In [2]:
import pandas as pd

# ===== 1) ĐƯỜNG DẪN FILE GỐC =====
file_path = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/04_djinni_round1_step1.xlsx"

# ===== 2) ĐỌC SHEET GỐC =====
df = pd.read_excel(file_path)

# ===== 3) LÀM SẠCH NHẸ =====
for col in ["ten_sach", "huong_xu_ly"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

# ===== 4) LỌC 3 NHÓM =====
df_xem_lai = df[df["huong_xu_ly"] == "xem_lai"].copy()
df_them_ten = df[df["huong_xu_ly"] == "them_ten_gan_giong"].copy()
df_doi_ten = df[df["huong_xu_ly"] == "doi_ten"].copy()

# ===== 5) BẢNG TỔNG QUAN =====
tong_quan = pd.DataFrame({
    "nhom": ["xem_lai", "them_ten_gan_giong", "doi_ten", "giu_nguyen", "trong"],
    "so_dong": [
        (df["huong_xu_ly"] == "xem_lai").sum(),
        (df["huong_xu_ly"] == "them_ten_gan_giong").sum(),
        (df["huong_xu_ly"] == "doi_ten").sum(),
        (df["huong_xu_ly"] == "giu_nguyen").sum(),
        (df["huong_xu_ly"] == "").sum()
    ]
})

# ===== 6) GHI THÊM SHEET VÀO CHÍNH FILE 04 =====
with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    tong_quan.to_excel(writer, sheet_name="tong_quan", index=False)
    df_xem_lai.to_excel(writer, sheet_name="ds_xem_lai", index=False)
    df_them_ten.to_excel(writer, sheet_name="ds_them_ten", index=False)
    df_doi_ten.to_excel(writer, sheet_name="ds_doi_ten", index=False)

print("Đã thêm sheet vào chính file 04:")
print("- tong_quan")
print("- ds_xem_lai")
print("- ds_them_ten")
print("- ds_doi_ten")
print()
print(tong_quan)

Đã thêm sheet vào chính file 04:
- tong_quan
- ds_xem_lai
- ds_them_ten
- ds_doi_ten

                 nhom  so_dong
0             xem_lai       24
1  them_ten_gan_giong       28
2             doi_ten        3
3          giu_nguyen       11
4               trong     1105


In [5]:
import pandas as pd

# ===== 1) ĐƯỜNG DẪN FILE =====
input_file = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/04_djinni_round1_step1.xlsx"
output_file = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/05_djinni_round1_finals.xlsx"

# ===== 2) ĐỌC SHEET GỐC =====
df_main = pd.read_excel(input_file)

# ===== 3) ĐỌC 3 SHEET ĐÃ CHỈNH =====
df_xem_lai = pd.read_excel(input_file, sheet_name="ds_xem_lai_loc")
df_them_ten = pd.read_excel(input_file, sheet_name="ds_them_ten_loc")
df_doi_ten = pd.read_excel(input_file, sheet_name="ds_doi_ten_loc")

# ===== 4) GHÉP 3 SHEET REVIEW =====
df_review = pd.concat([df_xem_lai, df_them_ten, df_doi_ten], ignore_index=True)

# ===== 5) CHUẨN HOÁ NHẸ =====
for df in [df_main, df_review]:
    for col in [
        "nhom_lon", "nhom_nho", "ten_goc", "ten_sach",
        "ten_thi_truong", "ten_gan_giong",
        "huong_xu_ly", "ghi_chu_djinni"
    ]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str).str.strip()

# ===== 6) TẠO KHÓA GHÉP =====
df_main["key"] = (
    df_main["nhom_lon"].str.lower() + " || " +
    df_main["nhom_nho"].str.lower() + " || " +
    df_main["ten_goc"].str.lower() + " || " +
    df_main["ten_sach"].str.lower()
)

df_review["key"] = (
    df_review["nhom_lon"].str.lower() + " || " +
    df_review["nhom_nho"].str.lower() + " || " +
    df_review["ten_goc"].str.lower() + " || " +
    df_review["ten_sach"].str.lower()
)

# ===== 7) TẠO MAP REVIEW =====
cot_cap_nhat = ["ten_thi_truong", "ten_gan_giong", "huong_xu_ly", "ghi_chu_djinni"]

review_map = (
    df_review[["key"] + cot_cap_nhat]
    .drop_duplicates(subset=["key"], keep="last")
    .set_index("key")
    .to_dict(orient="index")
)

# ===== 8) GHI ĐÈ CÁC DÒNG ĐÃ REVIEW =====
for i in df_main.index:
    key = df_main.at[i, "key"]
    if key in review_map:
        for col in cot_cap_nhat:
            gia_tri_moi = review_map[key].get(col, "")
            if str(gia_tri_moi).strip() != "":
                df_main.at[i, col] = gia_tri_moi

# ===== 9) XỬ LÝ CÁC DÒNG CÒN TRỐNG =====
# nếu chưa có hướng xử lý thì cho là giữ nguyên
df_main["huong_xu_ly"] = df_main["huong_xu_ly"].fillna("").astype(str).str.strip()
df_main.loc[df_main["huong_xu_ly"] == "", "huong_xu_ly"] = "giu_nguyen"

# nếu chưa có tên thị trường thì lấy từ tên sạch
df_main["ten_thi_truong"] = df_main["ten_thi_truong"].fillna("").astype(str).str.strip()
df_main["ten_sach"] = df_main["ten_sach"].fillna("").astype(str).str.strip()

mask_ten_thi_truong_trong = df_main["ten_thi_truong"] == ""
df_main.loc[mask_ten_thi_truong_trong, "ten_thi_truong"] = df_main.loc[mask_ten_thi_truong_trong, "ten_sach"]

# nếu chưa có tên gần giống thì để trống
if "ten_gan_giong" not in df_main.columns:
    df_main["ten_gan_giong"] = ""
df_main["ten_gan_giong"] = df_main["ten_gan_giong"].fillna("").astype(str).str.strip()

# ===== 10) XỬ NỐT DÒNG xem_lai CÒN LẠI (NẾU CÓ) =====
mask_info_system = df_main["ten_sach"].str.lower() == "keep up with the latest information systems solutions"
df_main.loc[mask_info_system, "ten_thi_truong"] = "information systems"
df_main.loc[mask_info_system, "ten_gan_giong"] = "enterprise solutions, business systems, it solutions"
df_main.loc[mask_info_system, "huong_xu_ly"] = "them_ten_gan_giong"

# ===== 11) BỎ CỘT PHỤ =====
df_main = df_main.drop(columns=["key"])

# ===== 12) LƯU FILE FINAL =====
df_main.to_excel(output_file, index=False)

print("Đã tạo file cuối:", output_file)
print()
print("Thống kê huong_xu_ly sau khi chốt:")
print(df_main["huong_xu_ly"].value_counts(dropna=False))
print()
print("Số ô trống ten_thi_truong:", (df_main["ten_thi_truong"].astype(str).str.strip() == "").sum())
print("Số dòng xem_lai còn lại:", (df_main["huong_xu_ly"] == "xem_lai").sum())

Đã tạo file cuối: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/05_djinni_round1_finals.xlsx

Thống kê huong_xu_ly sau khi chốt:
huong_xu_ly
giu_nguyen            1116
doi_ten                 46
them_ten_gan_giong       9
Name: count, dtype: int64

Số ô trống ten_thi_truong: 0
Số dòng xem_lai còn lại: 0


In [7]:
import pandas as pd

df = pd.read_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/05_djinni_round1_finals.xlsx")

print(df["huong_xu_ly"].fillna("").astype(str).str.strip().value_counts(dropna=False))
print("so dong trong huong_xu_ly:", (df["huong_xu_ly"].fillna("").astype(str).str.strip() == "").sum())
print("so dong xem_lai:", (df["huong_xu_ly"].fillna("").astype(str).str.strip() == "xem_lai").sum())

huong_xu_ly
giu_nguyen            1116
doi_ten                 46
them_ten_gan_giong       9
Name: count, dtype: int64
so dong trong huong_xu_ly: 0
so dong xem_lai: 0
